<a href="https://colab.research.google.com/github/Sudiptermux/INFOSYS-_INTERNSHIP-OIL_SPILL_DETECTION-/blob/main/SpillDetApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install streamlit ultralytics pandas plotly opencv-python requests streamlit-option-menu streamlit-lottie fpdf streamlit-webrtc av pyngrok -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.4/802.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.2/220.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.0 MB/s eta 0:00:00
ERROR: pip's

In [9]:
# Install the required library first
!pip install -q segmentation-models-pytorch

import segmentation_models_pytorch as smp

# This new code replaces your entire previous UNET class definition.
# It creates a U-Net with a powerful ResNet34 encoder pre-trained on ImageNet,
# which is the most effective way to improve your model's accuracy.
model = smp.Unet(
    encoder_name="resnet34",        # Use the popular and effective ResNet34 backbone
    encoder_weights="imagenet",     # Load weights pre-trained on ImageNet for a head start
    in_channels=3,                  # Your input images have 3 channels
    classes=1,                      # Your output is a single mask
)

print("U-Net with pre-trained ResNet34 encoder created successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

U-Net with pre-trained ResNet34 encoder created successfully.


In [3]:
%%writefile app.py
import streamlit as st
import torch
import segmentation_models_pytorch as smp
import torchvision.transforms as transforms
from PIL import Image, ImageDraw
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime
import os
import base64

# --- PAGE-SPECIFIC WALLPAPERS ---
def get_base64_image(image_path):
    if not os.path.exists(image_path): return ""
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode()

# Backgrounds for each page — change as you like!
WALLPAPER_HOME = "/content/drive/MyDrive/wallpaper2.jpg"
WALLPAPER_UPLOAD = "/content/drive/MyDrive/wallpaper1.png"
WALLPAPER_ANALYSIS = "/content/drive/MyDrive/wallpaper4.jpg"

wallpaper_dict = {
    "home": get_base64_image(WALLPAPER_HOME),
    "upload": get_base64_image(WALLPAPER_UPLOAD),
    "analysis": get_base64_image(WALLPAPER_ANALYSIS)
}

def inject_css(page="home"):
    base64_img = wallpaper_dict.get(page, wallpaper_dict["home"])
    st.markdown(
        f"""
        <style>
        .stApp {{
            background-image: url("data:image/jpg;base64,{base64_img}");
            background-size: cover;
            background-position: center;
            background-repeat: no-repeat;
            background-attachment: fixed;
            color: #e2ecf6 !important;
        }}
        .main-content {{
            background: rgba(22,28,48,0.83);
            padding: 2.2rem 2.5rem 1.2rem 2.5rem;
            border-radius: 24px;
            margin: 2.5rem auto;
            max-width: 850px;
            box-shadow: 0 8px 20px rgba(0,0,0,0.6);
        }}
        h1, h2, h3 {{
            color:#31cafb;
            text-align:center;
            letter-spacing:1.5px;
            font-weight:700;
        }}
        .nav-btn {{
            background: #fe4a49 !important;
            color: #fff !important;
            border: none !important;
            border-radius: 40px !important;
            font-weight: 700 !important;
            padding: 11px 37px !important;
            font-size: 19px !important;
            transition: background 0.17s;
            margin-bottom:0 !important;
            margin-left: 7px !important;
            margin-right: 7px !important;
            cursor: pointer;
            box-shadow: 0 10px 30px 0 rgba(16,30,60, 0.3);
        }}
        .nav-btn:hover, .nav-btn:active {{
            background: #95231a !important;
            color: #fff !important;
        }}
        div.stButton > button {{
            background-color: #fe4a49 !important;
            color: white !important;
            border-radius: 36px !important;
            font-weight: 700 !important;
            font-size: 19px !important;
            border: none !important;
            margin-bottom: 3px !important;
            margin-top: 3px !important;
            transition: background 0.16s;
            box-shadow: 0 6px 22px 0 rgba(200,60,60,0.12);
        }}
        div.stButton > button:hover, div.stButton > button:active {{
            background-color: #95231a !important;
            color: #fff !important;
        }}
        table td, table th {{
            color:#e2eefd !important;
        }}
        div[data-testid="stFileUploader"] section div {{
            color: #FFFACD !important;
            font-weight: 600 !important;
            font-size: 17px !important;
            letter-spacing: 1px;
        }}
        div[data-testid="stFileUploader"] svg {{
            color: #DB7093 !important;
        }}
        div[data-testid="stFileUploader"] label > div {{
            color: #FFFACD !important;
            font-weight: 800 !important;
            font-size: 19px !important;
            letter-spacing: 1px;
        }}
        div[data-testid="stFileUploader"] input[type="file"]::file-selector-button,
        div[data-testid="stFileUploader"] input[type="file"]::placeholder,
        div[data-testid="stFileUploader"] input[type="file"]:focus::file-selector-button {{
            color: #FFFACD !important;
        }}
        div[data-testid="stFileUploader"] button {{
            color: #5F9EA0 !important;
            font-weight: 700 !important;
        }}
        </style>
        """, unsafe_allow_html=True
    )


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/content/drive/MyDrive/best_model_custom_unet.pth.tar"

@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        st.error(f"Model not found at {MODEL_PATH}")
        return None
    model = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1)
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(DEVICE)
    model.eval()
    return model

def preprocess_image(img):
    t = transforms.Compose([
        transforms.Resize((256,256)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    return t(img).unsqueeze(0).to(DEVICE)

def postprocess_mask(mask_tensor, size):
    mask = mask_tensor.squeeze().detach().cpu().numpy()
    mask = (mask > 0.5).astype(np.uint8)*255
    return Image.fromarray(mask).resize(size)

def overlay_mask(image, mask):
    # Convert to matching size and numpy arrays
    image = image.convert("RGB")
    mask = mask.convert("L").resize(image.size)
    img_np = np.array(image)
    mask_np = np.array(mask)

    # Boolean mask: True where mask is 255 (spill area)
    spill_area = mask_np == 255

    # Create a copy; fill spill area with vivid color (fully opaque)
    overlay_np = img_np.copy()
    overlay_np[spill_area] = (252, 46, 130)  # vivid pink (fully opaque)

    # Convert back to PIL
    return Image.fromarray(overlay_np)


def run_inference(img, model):
    t_img = preprocess_image(img)
    with torch.no_grad():
        out = model(t_img)
    mask_img = postprocess_mask(out, img.size)
    overlay = overlay_mask(img, mask_img)
    spill = np.array(mask_img).astype(bool).sum()
    spill_pct = 100 * spill / (mask_img.size[0]*mask_img.size[1])
    return overlay, spill_pct

def init_state():
    if "page" not in st.session_state: st.session_state.page = "home"
    if "uploaded" not in st.session_state: st.session_state.uploaded = {}
    if "results" not in st.session_state: st.session_state.results = []

def nav_buttons(page):
    st.markdown('<br>', unsafe_allow_html=True)
    c1,c2 = st.columns([1,1])
    if c1.button("Back to Home", key=f"bth_{page}", use_container_width=True):
        st.session_state.page = "home"
        return  # Remove st.experimental_rerun()
    if page == "upload":
        if c2.button("Go to Analysis", key="goto_analysis", use_container_width=True):
            st.session_state.page = "analysis"
            return  # Remove st.experimental_rerun()
    elif page == "analysis":
        if c2.button("Go to Upload & Detect", key="goto_upload", use_container_width=True):
            st.session_state.page = "upload"
            return  # Remove st.experimental_rerun()

def clear_all():
    st.session_state.uploaded.clear()
    st.session_state.results.clear()
    st.success("Data cleared!")

def page_home():
    inject_css("home")
    st.markdown('<div class="main-content">', unsafe_allow_html=True)
    st.title("Premium Oil Spill Detection")
    st.markdown("""
        <p style="font-size:1.23rem;text-align:center;">
        Upload satellite images. Your oil spills are immediately highlighted with overlays after detection, and you can analyze all images.<br><br>
        <b>AI for environmental safety. Premium experience.</b>
        </p>
        """, unsafe_allow_html=True)
    st.markdown('<br>', unsafe_allow_html=True)
    c1, c2 = st.columns([1,1])
    if c1.button("Upload & Detect Page", key="homepage_goto_upload", use_container_width=True):
        st.session_state.page = "upload"
        return
    if c2.button("Analysis Page", key="homepage_goto_analysis", use_container_width=True):
        st.session_state.page = "analysis"
        return
    st.markdown("</div>", unsafe_allow_html=True)


def page_upload(model):
    inject_css("upload")
    st.markdown('<div class="main-content">', unsafe_allow_html=True)
    st.header("Upload & Detect Oil Spills")
    files = st.file_uploader("Select images for detection (JPG/JPEG/PNG)", type=['jpg','jpeg','png'], accept_multiple_files=True)
    threshold = st.slider("Alert threshold (%)",1,99,10)
    if files:
        for f in files:
            if f.name not in st.session_state.uploaded:
                st.session_state.uploaded[f.name] = Image.open(f).convert("RGB")
        if st.button("Run Detection", use_container_width=True):
            st.session_state.results.clear()
            with st.spinner("Detecting spills..."):
                for name,img in st.session_state.uploaded.items():
                    overlay, pct = run_inference(img, model)
                    alert = "Good" if pct < threshold else "Critical"
                    st.session_state.results.append({
                        "name":name, "original":img, "overlay":overlay, "spill_pct":pct,
                        "alert":alert, "time":datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })
            st.success("Detection finished! Go to analysis page.")
    if st.session_state.uploaded:
        st.write("Preview of uploaded images:")
        cols = st.columns(min(4,len(st.session_state.uploaded)))
        for i,(n,img) in enumerate(st.session_state.uploaded.items()):
            with cols[i%4]:
                st.image(img, caption=n)
    nav_buttons("upload")
    st.markdown("</div>", unsafe_allow_html=True)

def page_analysis():
    inject_css("analysis")
    st.markdown('<div class="main-content">', unsafe_allow_html=True)
    st.header("Detection Analysis")
    if not st.session_state.results:
        st.warning("No results yet. Please detect spills first.")
        nav_buttons("analysis")
        st.markdown('</div>', unsafe_allow_html=True)
        return
    df = pd.DataFrame([{
        "Image":r["name"], "Spill Area %":r["spill_pct"],
        "Alert":r["alert"], "Time":r["time"] } for r in st.session_state.results])
    st.dataframe(df)
    c1,c2,c3 = st.columns(3)
    c1.metric("Images", len(df))
    c2.metric("Avg Spill %", f"{df['Spill Area %'].mean():.2f}")
    c3.metric("Critical Alerts", (df['Alert']=='Critical').sum())
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df["Image"], y=df["Spill Area %"], marker_color="#31cafb"))
    fig.update_layout(title="Oil Spill % by Image", xaxis_title="Image", yaxis_title="Spill %", plot_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig, use_container_width=True)
    for r in st.session_state.results:
        st.image(r["overlay"], caption=f"{r['name']} (Oil Spill Overlay)")
    nav_buttons("analysis")
    st.markdown("</div>", unsafe_allow_html=True)

# --------------- MAIN APP ----------------------
init_state()
model = load_model()
if model is None:
    st.stop()
if st.session_state.page == "home":
    page_home()
elif st.session_state.page == "upload":
    page_upload(model)
elif st.session_state.page == "analysis":
    page_analysis()


Writing app.py


In [4]:
import os
MODEL_PATH = "/content/drive/MyDrive/best_model_custom_unet.pth.tar" # as used in your code

print("Current working directory:", os.getcwd())
print("Files in current directory:", os.listdir())
print("Does model path exist?:", os.path.exists(MODEL_PATH))


Current working directory: /content
Files in current directory: ['.config', 'drive', 'app.py', 'sample_data']
Does model path exist?: True


In [5]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 4s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [6]:
!wget -q -O - ipv4.icanhazip.com

35.188.186.93


In [7]:
import os
os.system("pkill -f streamlit")
os.system("pkill -f ngrok")

15

In [8]:
from pyngrok import ngrok
import time

# --- ACTION: Paste your ngrok authtoken here ---
authtoken = "34ShMKTzy4vUAfaAMnLcAClSIgD_2gd7Rfs9Q54df5YrvaPAv"
# ---

# Set up the authtoken
ngrok.kill()
ngrok.set_auth_token(authtoken)

# Run the streamlit app in the background
!nohup streamlit run app.py &

# Wait 5 seconds for the app to start
time.sleep(2)

# Create a public URL to the app on port 8501
public_url = ngrok.connect(8501)
print("✅ Your app is live! Click this link:")
print(public_url)

nohup: appending output to 'nohup.out'
✅ Your app is live! Click this link:
NgrokTunnel: "https://unhumbled-digna-unremitting.ngrok-free.dev" -> "http://localhost:8501"
